In [1]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
isin = pd.read_excel('Data\\EA_ISINs.xlsx')

In [3]:
unique_isin = tuple(isin['ISIN'])

In [4]:
isin['ISIN'].str[:2].unique()

array(['DE', 'IT', 'FR', 'ES'], dtype=object)

In [5]:
treasury = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [6]:
unique_treasury = tuple(treasury['ISIN'].unique())

In [7]:
hedge_funds = pd.read_csv('key dataframe\\overlap_hedge_funds.csv')

In [8]:
hf_overlap = tuple(hedge_funds['entity_id'].unique())

In [9]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    s.security_isin,
    SUM(s.nominal_value)                                                                AS borrowing_volume,
    AVG(repo_rate)                                                                      AS borrowing_rate, 
    AVG(CASE WHEN contractual_maturity < 1 THEN 1 ELSE contractual_maturity END)        AS borrowing_term
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND s_lender.sector = 'DEALER'
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id, s.security_isin
ORDER BY s.business_date, s.borrower_id, s.lender_id, s.security_isin

"""

df_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_33640\667081715.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing = pd.read_sql_query(query, cnxn)


In [10]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    s.security_isin,
    SUM(s.nominal_value)                                                                AS lending_volume,
    AVG(repo_rate)                                                                      AS lending_rate, 
    AVG(CASE WHEN contractual_maturity < 1 THEN 1 ELSE contractual_maturity END)        AS lending_term
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND s_borrower.sector = 'DEALER'
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id, s.security_isin
ORDER BY s.business_date, s.lender_id, s.borrower_id, s.security_isin
"""

df_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_33640\2678799833.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lending = pd.read_sql_query(query, cnxn)


In [11]:
df = df_borrowing.merge(df_lending, on= ['business_date', 'fund_id', 'dealer_id', 'security_isin'], how = 'outer')

In [14]:
df.to_csv('key dataframe\\fund_dealer_isin_day.csv')

In [9]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS dealer_id,
    LEFT(s.security_isin,2)                                                             AS collateral_country,
    SUM(s.nominal_value)                                                                AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'DEALER'
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, collateral_country
ORDER BY s.business_date, s.borrower_id, collateral_country
"""

df_dealer_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_27036\1100536720.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dealer_borrowing = pd.read_sql_query(query, cnxn)


In [10]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS dealer_id,
    LEFT(s.security_isin,2)                                                             AS collateral_country,
    SUM(s.nominal_value)                                                                AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'DEALER'
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, collateral_country
ORDER BY s.business_date, s.lender_id, collateral_country
"""

df_dealer_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_27036\227726236.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dealer_lending = pd.read_sql_query(query, cnxn)


In [11]:
df_dealer = df_dealer_borrowing.merge(df_dealer_lending, on= ['business_date', 'dealer_id', 'collateral_country'], how = 'outer')

In [13]:
df_dealer.to_csv('key dataframe\\dealer_country_day.csv')